<a href="https://colab.research.google.com/github/AV2706/GDGSC-Study-Jam/blob/AnantAgarwal07/YOLOv8_Forensic_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install ultralytics tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 55.2 MB/s eta 0:00:00


In [4]:
from ultralytics import YOLO

# Load YOLOv8 small model
model = YOLO("yolov8s.pt")

# Export to TensorFlow SavedModel format
model.export(format="saved_model")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.6 🚀 Python-3.12.12 torch-2.9.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLOv8s summary (fused): 72 layers, 11,156,544 parameters, 0 gradients, 28.6 GFLOPs

PyTorch: starting from 'yolov8s.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (21.5 MB)
requirements: Ultralytics requirements ['sng4onnx>=1.0.1', 'onnx_graphsurgeon>=0.3.26', 'ai-edge-litert>=1.2.0', 'onnx>=1.12.0,<2.0.0', 'onnx2tf>=1.26.3', 'onnxslim>=0.1.71', 'onnxruntime'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at

/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/utils.py:1447: OnnxExporterWarning: Exporting to ONNX opset version 22 is not supported. by 'torch.onnx.export()'. The highest opset version supported is 20. To use a newer opset version, consider 'torch.onnx.export(..., dynamo=True)'. 
  warnings.warn(


ONNX: slimming with onnxslim 0.1.82...
ONNX: export success ✅ 2.5s, saved as 'yolov8s.onnx' (42.8 MB)
Unzipping calibration_image_sample_data_20x128x128x3_float32.npy.zip to /content/calibration_image_sample_data_20x128x128x3_float32.npy...: 100% ━━━━━━━━━━━━ 1/1 41.3files/s 0.0s
TensorFlow SavedModel: starting TFLite export with onnx2tf 1.28.8...
Saved artifact at 'yolov8s_saved_model'. The following endpoints are available:

* Endpoint 'serving_default'
  inputs_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 640, 640, 3), dtype=tf.float32, name='images')
Output Type:
  TensorSpec(shape=(1, 84, 8400), dtype=tf.float32, name=None)
Captures:
  132939214124432: TensorSpec(shape=(4, 2), dtype=tf.int32, name=None)
  132939214122896: TensorSpec(shape=(3, 3, 3, 32), dtype=tf.float32, name=None)
  132939214123664: TensorSpec(shape=(32,), dtype=tf.float32, name=None)
  132939115382736: TensorSpec(shape=(4, 2), dtype=tf.int32, name=None)
  132939214124240: TensorSpec(shape=(3, 3, 32, 64), dtype=tf.f

'yolov8s_saved_model'

In [48]:
!wget -q -O evidence.jpeg "https://imgs.search.brave.com/wrzmrxKcjd3xlyNM2LU17hNk5j8AOcBwNcJ9syfkOZk/rs:fit:500:0:1:0/g:ce/aHR0cHM6Ly9jZWxs/ZWJyaXRlLmNvbS93/cC1jb250ZW50L3Vw/bG9hZHMvMjAyMC8w/My9Ib3ctdG8tUHJv/cGVybHktSGFuZGxl/LVBob25lcy1TZWl6/ZWQtZm9yLUludmVz/dGlnYXRpb24tUDEt/MTAyNHg1MDQuanBn"
results = model("evidence.jpeg")

detections = results[0].boxes
class_names = results[0].names


image 1/1 /content/evidence.jpeg: 320x640 1 person, 1 laptop, 1 keyboard, 1 cell phone, 325.6ms
Speed: 2.5ms preprocess, 325.6ms inference, 1.1ms postprocess per image at shape (1, 3, 320, 640)


In [49]:
FORENSIC_EVIDENCE = ["cell phone", "knife", "laptop","gun"]

evidence_found = {}
is_potential_evidence = False

for box in detections:
    cls_id = int(box.cls)
    confidence = float(box.conf)
    label = class_names[cls_id]

    if label in FORENSIC_EVIDENCE:
        if label not in evidence_found or confidence > evidence_found[label]:
            evidence_found[label] = round(confidence, 2)
            is_potential_evidence = True



In [51]:
import json

forensic_report = {
    "detected_objects": list(evidence_found.keys()),
    "confidence_scores": evidence_found,
    "is_potential_evidence": is_potential_evidence
}

print("Forensic Scan Result:")
print(json.dumps(forensic_report, indent=4))


Forensic Scan Result:
{
    "detected_objects": [
        "cell phone",
        "laptop"
    ],
    "confidence_scores": {
        "cell phone": 0.9,
        "laptop": 0.66
    },
    "is_potential_evidence": true
}
